## Reproducing Figure 2d and 2e

This builds off the reproduction of Figures 2b and c from the reproduction.ipynb notebook.

Things to keep in mind before running: 
1. Cell type name mismatch (Cell 8): After running Cell 7, look at abund.columns.tolist() and update the dictionary in Cell 8 to match your exact strings. This is the most likely single point of failure. If none match, use the fallback in Cell 9.

2. results_spatial['adata'] key: If the old API returns something different (some versions use a list, some a dict), wrap Cell 5's extraction in print(type(results_spatial)) and print(results_spatial.keys() if isinstance(results_spatial, dict) else len(results_spatial)) to debug.

3. Spatial coordinates lost: The old run_cell2location sometimes drops .uns['spatial'] and .obsm['spatial']. Cell 6 restores them from the original adata_vis — make sure adata_vis is still in scope when you run Cell 6.

4. For a fast test: Change n_epochs in Cell 5 to 3000 first. If the plot looks anatomically plausible (neurons in cortex, not random noise), then run 30,000 for the final figure.


In [1]:
# Disable autosave polling to stop the popup
from IPython.display import Javascript
display(Javascript("IPython.notebook.set_autosave_interval(0)"))

<IPython.core.display.Javascript object>

In [2]:
# ══════════════════════════════════════════════════════════════════
# CELL 1 — Imports & Paths
# ══════════════════════════════════════════════════════════════════
import os
os.environ["THEANO_FLAGS"] = "device=cuda0,contexts=dev0->cuda0,floatX=float32"

import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')          # non-interactive backend for SCC
import matplotlib.pyplot as plt
import cell2location
import torch
import theano

data_dir    = "/projectnb/ds596/projects/Team_7/data/"
scrna_dir   = os.path.join(data_dir, "mousescRNAseq")
visium_dir  = os.path.join(data_dir, "mouseVisium")
reg_dir     = "/projectnb/ds596/projects/Team_7/results_regression_v2/"
spatial_dir = "/projectnb/ds596/projects/Team_7/results_spatial_v2/"
os.makedirs(spatial_dir, exist_ok=True)

print("Theano device:", theano.config.device)
print("cell2location imported successfully!")
print("GPU available:", torch.cuda.is_available())

/opt/conda/envs/cellpymc/lib/python3.7/site-packages/theano/gpuarray/dnn.py:184: UserWarning: Your cuDNN version is more recent than Theano. If you encounter problems, try updating Theano or downgrading cuDNN to a version >= v5 and <= v7.
  warnings.warn("Your cuDNN version is more recent than "
Using cuDNN version 7605 on context None
Mapped name None to device cuda0: Tesla V100-SXM2-16GB (0000:AF:00.0)
Mapped name dev0 to device cuda0: Tesla V100-SXM2-16GB (0000:AF:00.0)


Theano device: cuda0
cell2location imported successfully!
GPU available: True


In [3]:
# ══════════════════════════════════════════════════════════════════
# CELL 2 — Recover inf_aver from saved regression (skip if re-running regression)
# This loads the output you already have on disk.
# ══════════════════════════════════════════════════════════════════

# Find the regression subfolder automatically
reg_subfolders = [d for d in os.listdir(reg_dir) if d.startswith("Regression")]
assert len(reg_subfolders) == 1, f"Expected 1 regression folder, found: {reg_subfolders}"
reg_path = os.path.join(reg_dir, reg_subfolders[0], "sc.h5ad")
print(f"Loading regression result from:\n  {reg_path}")

adata_reg = sc.read(reg_path)
reg_mod   = adata_reg.uns['regression_mod']

print("fact_names (first 5):", reg_mod['fact_names'][:5])
print("post_sample_means keys:", list(reg_mod['post_sample_means'].keys()))

# Build inf_aver — genes × cell_types matrix of expected expression
inf_aver = pd.DataFrame(
    reg_mod['post_sample_means']['gene_factors'].T,
    index   = adata_reg.var_names,
    columns = reg_mod['fact_names']
)
print(f"\ninf_aver shape (genes × cell types): {inf_aver.shape}")
print("Cell types:", inf_aver.columns.tolist())

Loading regression result from:
  /projectnb/ds596/projects/Team_7/results_regression_v2/RegressionGeneBackgroundCoverageTorch_65covariates_40532cells_10085genes/sc.h5ad
fact_names (first 5): ['sample_5705STDY8058280' 'sample_5705STDY8058281'
 'sample_5705STDY8058282' 'sample_5705STDY8058283'
 'sample_5705STDY8058284']
post_sample_means keys: ['gene_E', 'gene_factors', 'sample_scaling']

inf_aver shape (genes × cell types): (10085, 65)
Cell types: ['sample_5705STDY8058280', 'sample_5705STDY8058281', 'sample_5705STDY8058282', 'sample_5705STDY8058283', 'sample_5705STDY8058284', 'sample_5705STDY8058285', 'annotation_1_Astro_AMY', 'annotation_1_Astro_AMY_CTX', 'annotation_1_Astro_CTX', 'annotation_1_Astro_HPC', 'annotation_1_Astro_HYPO', 'annotation_1_Astro_STR', 'annotation_1_Astro_THAL_hab', 'annotation_1_Astro_THAL_lat', 'annotation_1_Astro_THAL_med', 'annotation_1_Astro_WM', 'annotation_1_Endo', 'annotation_1_Ext_Amy_1', 'annotation_1_Ext_Amy_2', 'annotation_1_Ext_ClauPyr', 'annotation

In [4]:
# ══════════════════════════════════════════════════════════════════
# CELL 3 — Load ONE Visium section with raw counts
# Use ST8059048 — the first section. Do NOT normalize.
# ══════════════════════════════════════════════════════════════════
sample_key = "ST8059048"

adata_vis = sc.read_visium(
    path       = os.path.join(visium_dir, f"{sample_key}_spatial"),
    count_file = os.path.join(visium_dir, f"{sample_key}_filtered_feature_bc_matrix.h5"),
    load_images = True,
)
adata_vis.var_names_make_unique()
adata_vis.obs["sample"] = sample_key

# CRITICAL: store raw counts before anything else touches .X
adata_vis.layers["counts"] = adata_vis.X.copy()

# Basic QC — flag mito genes but don't filter spots for mapping
adata_vis.var["mt"] = adata_vis.var_names.str.startswith("mt-")
sc.pp.calculate_qc_metrics(adata_vis, qc_vars=["mt"], inplace=True)

print(f"Visium spots: {adata_vis.n_obs}, genes: {adata_vis.n_vars}")

# ── Intersect genes between Visium and the regression reference ──
shared_genes    = np.intersect1d(adata_vis.var_names, inf_aver.index)
adata_vis       = adata_vis[:, shared_genes].copy()
inf_aver_shared = inf_aver.loc[shared_genes, :]

print(f"Shared genes: {len(shared_genes)}")
print(f"inf_aver for mapping: {inf_aver_shared.shape}")
# Expect ~8,000-10,000 shared genes — if much lower, check gene name format (Ensembl vs symbol)

Variable names are not unique. To make them unique, call `.var_names_make_unique`.
/opt/conda/envs/cellpymc/lib/python3.7/site-packages/anndata/_core/anndata.py:1094: FutureWarning: is_categorical is deprecated and will be removed in a future version.  Use is_categorical_dtype instead
Variable names are not unique. To make them unique, call `.var_names_make_unique`.


Visium spots: 2987, genes: 31053
Shared genes: 10085
inf_aver for mapping: (10085, 65)


In [5]:
# ══════════════════════════════════════════════════════════════════
# CELL 4 — Set .raw (REQUIRED by old cell2location API)
# run_cell2location reads sp_data.raw.X — this must be raw integer counts.
# ══════════════════════════════════════════════════════════════════

# At this point adata_vis.X is still raw (we never normalized it), so this is correct.
adata_vis.raw = adata_vis

print(f".raw.X shape: {adata_vis.raw.X.shape}")
print(f"First 5 values of spot 0 (should be integers): {adata_vis.raw.X[0, :5]}")

.raw.X shape: (2987, 10085)
First 5 values of spot 0 (should be integers):   (0, 3)	1.0
  (0, 0)	1.0
  (0, 2)	2.0


In [6]:
#Diagnostic cell
import torch
print("GPU available:", torch.cuda.is_available())
print("GPU device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

GPU available: True
GPU device: Tesla V100-SXM2-16GB


In [7]:
# ══════════════════════════════════════════════════════════════════
# CELL 5 — Run spatial mapping (OLD API — matches your singularity container)
#
# Key parameters matching the paper:
#   cells_per_spot   = 8   (N_cells_per_location in paper Methods)
#   factors_per_spot = 7   (expected distinct cell types per spot)
#   combs_per_spot   = 2.5 (expected co-occurring combinations)
#   n_epochs         = 30000 (paper value; use 3000 for a quick test first)
# ══════════════════════════════════════════════════════════════════
use_gpu = torch.cuda.is_available()
print(f"Running on GPU: {use_gpu}")

results_spatial = cell2location.run_cell2location(
    sc_data  = inf_aver_shared,
    sp_data  = adata_vis,
    train_args = {
        'n_iter'     : 30000,   # ← set to 3000 for a fast test run first!
        'learning_rate': 0.001, # lowered to 0.001 for stability
        'use_cuda'     : use_gpu,
        'use_raw'      : True,    # reads from adata_vis.raw.X
    },
    export_args = {
        'path'       : spatial_dir,
        'save_model' : False,
    },
    model_kwargs = {
        'cell_number_prior': {
            'cells_per_spot'   : 8,
            'factors_per_spot' : 7,
            'combs_per_spot'   : 2.5,
        }
    }
)

print("Training finished!")

Running on GPU: True
### Summarising single cell clusters ###
### Creating model ### - time 0.01 min
### Analysis name: LocationModelLinearDependentW_65clusters_2987locations_10085genes
### Training model ###


Finished [100%]: Average Loss = 2.8112e+07


Finished [100%]: Average Loss = 2.8112e+07


/opt/conda/envs/cellpymc/lib/python3.7/site-packages/cell2location/models/pymc3_model.py:446: MatplotlibDeprecationWarning: Passing non-integers as three-element position specification is deprecated since 3.3 and will be removed two minor releases later.
/opt/conda/envs/cellpymc/lib/python3.7/site-packages/cell2location/models/pymc3_model.py:447: MatplotlibDeprecationWarning: Adding an axes using the same arguments as a previous axes currently reuses the earlier instance.  In a future version, a new instance will always be created and returned.  Meanwhile, this warning can be suppressed, and the future behavior ensured, by passing a unique label to each axes instance.


### Sampling posterior ### - time 19.46 min



### Saving results ###


/opt/conda/envs/cellpymc/lib/python3.7/site-packages/anndata/_core/anndata.py:1192: FutureWarning: is_categorical is deprecated and will be removed in a future version.  Use is_categorical_dtype instead
... storing 'sample' as categorical
... storing 'feature_types' as categorical
... storing 'genome' as categorical
... storing 'feature_types' as categorical
... storing 'genome' as categorical


### Ploting results ###
### Plotting posterior of W / cell locations ###
Some error in plotting with scanpy or `cell2location.plt.plot_factor_spatial()`
 IndexError('index 0 is out of bounds for axis 0 with size 0')
### Done ### - time 21.97 min
Training finished!


In [8]:
# ══════════════════════════════════════════════════════════════════
# Cell 5.5: Extract and save the result
# ══════════════════════════════════════════════════════════════════

# 1. Create the base AnnData object from the model
adata_vis_mapped = results_spatial['mod'].export2adata(adata_vis)

# 2. Extract the actual cell abundance results (the 'means')
abundance_results = results_spatial['mod'].spot_factors_df.copy()

# 3. Add the prefix that the rest of the notebook/plotting expects
abundance_results.columns = [f"means_cell_abundance_w_sf_{c}" for c in abundance_results.columns]

# 4. Save into the .obsm slot and verify
adata_vis_mapped.obsm['means_cell_abundance_w_sf'] = abundance_results

# Save to disk
adata_vis_mapped.write("results_30k_reproduction.h5ad")

print("Extraction successful!")
print("Resulting obsm keys:", adata_vis_mapped.obsm.keys())
print("Abundances shape:", adata_vis_mapped.obsm['means_cell_abundance_w_sf'].shape)
print("Data extracted, file saved as results_30k_reproduction.h5ad.")

/opt/conda/envs/cellpymc/lib/python3.7/site-packages/anndata/_core/anndata.py:1192: FutureWarning: is_categorical is deprecated and will be removed in a future version.  Use is_categorical_dtype instead
... storing 'sample' as categorical
... storing 'feature_types' as categorical
... storing 'genome' as categorical
... storing 'feature_types' as categorical
... storing 'genome' as categorical


Extraction successful!
Resulting obsm keys: KeysView(AxisArrays with keys: spatial, means_cell_abundance_w_sf)
Abundances shape: (2987, 65)
Data extracted, file saved as results_30k_reproduction.h5ad.


In [9]:
# ══════════════════════════════════════════════════════════════════
# CELL 6 — Extract abundances & save
# ══════════════════════════════════════════════════════════════════

# Determine the correct obsm key (differs slightly between old API versions)
candidate_keys = [k for k in adata_vis_mapped.obsm.keys() if 'means' in k.lower()]
print("Candidate abundance keys:", candidate_keys)
abund_key = candidate_keys[0]   # take the first means key

abund = pd.DataFrame(
    adata_vis_mapped.obsm[abund_key],
    index   = adata_vis_mapped.obs_names,
    columns = inf_aver_shared.columns,
)

# Add all cell types as columns in .obs for sc.pl.spatial
for col in abund.columns:
    adata_vis_mapped.obs[col] = abund[col].values

# Restore spatial info (sometimes lost in old API round-trip)
if 'spatial' not in adata_vis_mapped.obsm:
    adata_vis_mapped.obsm['spatial'] = adata_vis.obsm['spatial']
if 'spatial' not in adata_vis_mapped.uns:
    adata_vis_mapped.uns['spatial']  = adata_vis.uns['spatial']

# Save for reuse — so you don't need to re-run mapping
adata_vis_mapped.write(os.path.join(spatial_dir, "adata_vis_mapped.h5ad"))
print("Saved mapped AnnData.")
print(f"Abundances shape: {abund.shape}")

Candidate abundance keys: ['means_cell_abundance_w_sf']
Saved mapped AnnData.
Abundances shape: (2987, 65)


In [10]:
# ══════════════════════════════════════════════════════════════════
# CELL 7: Check what cell types you actually have ───────────────
print("Available cell types in your results:")
print(abund.columns.tolist())

Available cell types in your results:
['sample_5705STDY8058280', 'sample_5705STDY8058281', 'sample_5705STDY8058282', 'sample_5705STDY8058283', 'sample_5705STDY8058284', 'sample_5705STDY8058285', 'annotation_1_Astro_AMY', 'annotation_1_Astro_AMY_CTX', 'annotation_1_Astro_CTX', 'annotation_1_Astro_HPC', 'annotation_1_Astro_HYPO', 'annotation_1_Astro_STR', 'annotation_1_Astro_THAL_hab', 'annotation_1_Astro_THAL_lat', 'annotation_1_Astro_THAL_med', 'annotation_1_Astro_WM', 'annotation_1_Endo', 'annotation_1_Ext_Amy_1', 'annotation_1_Ext_Amy_2', 'annotation_1_Ext_ClauPyr', 'annotation_1_Ext_Hpc_CA1', 'annotation_1_Ext_Hpc_CA2', 'annotation_1_Ext_Hpc_CA3', 'annotation_1_Ext_Hpc_DG1', 'annotation_1_Ext_Hpc_DG2', 'annotation_1_Ext_L5_1', 'annotation_1_Ext_L5_2', 'annotation_1_Ext_L5_3', 'annotation_1_Ext_L6', 'annotation_1_Ext_L6B', 'annotation_1_Ext_L23', 'annotation_1_Ext_L25', 'annotation_1_Ext_L56', 'annotation_1_Ext_Med', 'annotation_1_Ext_Pir', 'annotation_1_Ext_Thal_1', 'annotation_1_Ex

In [11]:
# ══════════════════════════════════════════════════════════════════
# CELL 8 — Extract Target Cell Types for Figure 2d
# ══════════════════════════════════════════════════════════════════

# Updated dictionary based on Figure 2d from the paper
target_cell_types = {
    'Oligo_2': 'Oligodendrocytes (Oligo2)',
    'Inh_Meis2_3': 'Inhibitory (Meis2_3)',
    'Inh_4': 'Inhibitory (Inh4)',
    'Ext_Thal_1': 'Excitatory (Thalamus)',
    'Ext_L23': 'Excitatory (L2/3)',
    'Ext_L56': 'Excitatory (L5/6)'
}

# Run the search to find these in your 30k results
raw_results = results_spatial['mod'].spot_factors_df
final_2d_names = []

for tech_id, friendly_name in target_cell_types.items():
    # Find the column that contains the tech_id (e.g., 'Oligo_2')
    match = [c for c in raw_results.columns if tech_id in c]
    
    if match:
        tech_name = match[0]
        # Align and move to .obs
        adata_vis_mapped.obs[friendly_name] = raw_results[tech_name].reindex(adata_vis_mapped.obs_names).values.astype(float)
        final_2d_names.append(friendly_name)
        print(f"Found {tech_id}! Mapped to: {friendly_name}")
    else:
        print(f"Warning: {tech_id} not found in model results. Check Cell 7 list.")

print(f"\nReady to plot {len(final_2d_names)} cell types for Figure 2d.")

Found Oligo_2! Mapped to: Oligodendrocytes (Oligo2)
Found Inh_Meis2_3! Mapped to: Inhibitory (Meis2_3)
Found Inh_4! Mapped to: Inhibitory (Inh4)
Found Ext_Thal_1! Mapped to: Excitatory (Thalamus)
Found Ext_L23! Mapped to: Excitatory (L2/3)
Found Ext_L56! Mapped to: Excitatory (L5/6)

Ready to plot 6 cell types for Figure 2d.


In [12]:
# ══════════════════════════════════════════════════════════════════
# CELL 9 - PLOTTING FIGURE 2d (Spatial Map of Major Cell Types)
# ══════════════════════════════════════════════════════════════════

sc.pl.spatial(
    adata_vis_mapped,
    color = final_2d_names,
    ncols = 3,           # Creates a 2x3 grid
    img_key = "hires", 
    color_map = "magma",
    size = 1.3,
    vmin = 0,
    vmax = 'p99', 
    frameon = False,
    show = False
)

plt.savefig("figure_2d_30kiter.pdf", bbox_inches="tight", dpi=300)
print("Saved reproduction of Figure 2d!")

Saved reproduction of Figure 2d!


In [13]:
# ══════════════════════════════════════════════════════════════════
# CELL 10 - PLOTTING FIGURE 2e (Spatial Map of Sparse Inhibitory Neurons)
# ══════════════════════════════════════════════════════════════════

# 1. Make a dictionary for inhibitory neurons
inhibitory_targets = ['Inh_Sst', 'Inh_Lamp5', 'Inh_Vip'] 
raw_results = results_spatial['mod'].spot_factors_df

# 2. The loop will find it automatically
final_inh_names = []
for target in inhibitory_targets:
    match = [c for c in raw_results.columns if target in c]
    
    if match:
        tech_name = match[0]
        clean_name = target.replace('annotation_1_', '') 
        adata_vis_mapped.obs[clean_name] = raw_results[tech_name].reindex(adata_vis_mapped.obs_names).values.astype(float)
        final_inh_names.append(clean_name)
        print(f"Found {target}! Max value: {adata_vis_mapped.obs[clean_name].max():.2f}")
    else:
        print(f"Warning: Could not find any column containing '{target}'")

# 3. Plot
sc.pl.spatial(
    adata_vis_mapped,
    color = final_inh_names,
    ncols = 3,
    img_key = "hires", 
    color_map = "magma",
    size = 1.5,
    vmin = 0,
    vmax = 'p99', 
    frameon = False,
    show = False
)
plt.savefig("figure_2e_30kiter.pdf", bbox_inches="tight", dpi=300)
print("Saved reproduction of figure 2e!")

Found Inh_Sst! Max value: 4145.35
Found Inh_Lamp5! Max value: 3653.02
Found Inh_Vip! Max value: 4556.29


/opt/conda/envs/cellpymc/lib/python3.7/site-packages/anndata/_core/anndata.py:1192: FutureWarning: is_categorical is deprecated and will be removed in a future version.  Use is_categorical_dtype instead


Saved reproduction of figure 2e!
